### Imports

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Pandas:", pd.__version__)

Pandas: 3.0.5


### Locate the source

In [3]:
current_dir = Path.cwd()

possible_paths = [
    current_dir / "data" / "raw" / "online_retail_II.xlsx",
    current_dir.parent / "data" / "raw" / "online_retail_II.xlsx",
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path.resolve()
        break

print("Source file:")
print(DATA_PATH)

Source file:
C:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\data\raw\online_retail_II.xlsx


### Load both worksheets

In [4]:
excel_file = pd.ExcelFile(DATA_PATH)

frames = []

for sheet in excel_file.sheet_names:

    temp = pd.read_excel(
        DATA_PATH,
        sheet_name=sheet
    )

    temp["SourceSheet"] = sheet

    frames.append(temp)

raw_df = pd.concat(
    frames,
    ignore_index=True
)

print(f"Raw rows: {len(raw_df):,}")
print(f"Raw columns: {raw_df.shape[1]}")

Raw rows: 1,067,371
Raw columns: 9


### Preserve raw data and standardize column names

In [5]:
df = raw_df.copy()

df = df.rename(
    columns={
        "Invoice": "invoice",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "Price": "unit_price",
        "Customer ID": "customer_id",
        "Country": "country",
        "SourceSheet": "source_sheet"
    }
)

print(df.columns.tolist())

['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'source_sheet']


### Standardize datatypes

In [6]:
df["invoice"] = (
    df["invoice"]
    .astype("string")
    .str.strip()
)

df["stock_code"] = (
    df["stock_code"]
    .astype("string")
    .str.strip()
)

df["description"] = (
    df["description"]
    .astype("string")
    .str.strip()
)

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

df["source_sheet"] = (
    df["source_sheet"]
    .astype("string")
)

df["customer_id"] = (
    pd.to_numeric(
        df["customer_id"],
        errors="coerce"
    )
    .astype("Int64")
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

df["unit_price"] = pd.to_numeric(
    df["unit_price"],
    errors="coerce"
)

df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

print(df.dtypes)

invoice                 string
stock_code              string
description             string
quantity                 int64
invoice_date    datetime64[us]
unit_price             float64
customer_id              Int64
country                 string
source_sheet            string
dtype: object


### Add calendar fields

In [7]:
df["invoice_year"] = df["invoice_date"].dt.year
df["invoice_month"] = df["invoice_date"].dt.month
df["invoice_day"] = df["invoice_date"].dt.day

df["year_month"] = (
    df["invoice_date"]
    .dt.to_period("M")
    .astype("string")
)

df["invoice_week"] = (
    df["invoice_date"]
    .dt.to_period("W")
    .astype("string")
)

df["day_of_week"] = (
    df["invoice_date"]
    .dt.day_name()
)

print(df[
    [
        "invoice_date",
        "invoice_year",
        "invoice_month",
        "year_month",
        "invoice_week",
        "day_of_week"
    ]
].head())

         invoice_date  invoice_year  invoice_month year_month  \
0 2009-12-01 07:45:00          2009             12    2009-12   
1 2009-12-01 07:45:00          2009             12    2009-12   
2 2009-12-01 07:45:00          2009             12    2009-12   
3 2009-12-01 07:45:00          2009             12    2009-12   
4 2009-12-01 07:45:00          2009             12    2009-12   

            invoice_week day_of_week  
0  2009-11-30/2009-12-06     Tuesday  
1  2009-11-30/2009-12-06     Tuesday  
2  2009-11-30/2009-12-06     Tuesday  
3  2009-11-30/2009-12-06     Tuesday  
4  2009-11-30/2009-12-06     Tuesday  


### Add data-quality flags

In [8]:
df["is_cancellation"] = (
    df["invoice"]
    .str.startswith("C", na=False)
)

df["is_negative_quantity"] = (
    df["quantity"] < 0
)

df["is_zero_price"] = (
    df["unit_price"] == 0
)

df["is_negative_price"] = (
    df["unit_price"] < 0
)

df["is_missing_customer"] = (
    df["customer_id"].isna()
)

df["is_missing_description"] = (
    df["description"].isna()
)

In [9]:
duplicate_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country"
]

df["is_exact_duplicate"] = (
    df.duplicated(
        subset=duplicate_columns,
        keep="first"
    )
)

### Create raw transaction value

In [10]:
df["transaction_value"] = (
    df["quantity"] *
    df["unit_price"]
)

### Define known non-merchandise codes

In [11]:
financial_codes = {
    "B",
    "BANK CHARGES",
    "AMAZONFEE"
}

service_codes = {
    "POST",
    "DOT",
    "C2"
}

adjustment_codes = {
    "M",
    "ADJUST",
    "ADJUST2"
}

discount_codes = {
    "D"
}

sample_codes = {
    "S"
}

### Build transaction classification

In [12]:
conditions = [

    # Financial/non-product activity
    df["stock_code"].isin(financial_codes),

    # Shipping/carriage-related activity
    df["stock_code"].isin(service_codes),

    # Manual/system adjustments
    df["stock_code"].isin(adjustment_codes),

    # Explicit discounts
    df["stock_code"].isin(discount_codes),

    # Samples
    df["stock_code"].isin(sample_codes),

    # Test transactions
    df["stock_code"].str.startswith(
        "TEST",
        na=False
    ),

    # Customer merchandise cancellation/return
    (
        df["is_cancellation"] &
        (df["quantity"] < 0) &
        (df["unit_price"] > 0)
    ),

    # Negative stock movement without C invoice
    (
        (~df["is_cancellation"]) &
        (df["quantity"] < 0)
    ),

    # Free/zero-price physical product movement
    (
        (df["quantity"] > 0) &
        (df["unit_price"] == 0)
    ),

    # Standard merchandise sale
    (
        (df["quantity"] > 0) &
        (df["unit_price"] > 0)
    )
]

choices = [

    "Financial Adjustment",
    "Shipping / Service Charge",
    "Manual / System Adjustment",
    "Discount",
    "Sample",
    "Test Transaction",
    "Merchandise Cancellation",
    "Inventory / Operational Adjustment",
    "Zero-Price Product Movement",
    "Merchandise Sale"
]

df["transaction_type"] = np.select(
    conditions,
    choices,
    default="Other / Review"
)

### Check transaction classification

In [13]:
transaction_type_summary = (
    df["transaction_type"]
    .value_counts(dropna=False)
    .rename_axis("transaction_type")
    .reset_index(name="rows")
)

transaction_type_summary["percentage"] = (
    transaction_type_summary["rows"]
    / len(df)
    * 100
).round(2)

display(transaction_type_summary)

,transaction_type,rows,percentage
0,Merchandise Sale,1037103,97.16
1,Merchandise Cancellation,18303,1.71
2,Shipping / Service Charge,3850,0.36
3,Inventory / Operational Adjustment,3457,0.32
4,Zero-Price Product Movement,2718,0.25
5,Manual / System Adjustment,1491,0.14
6,Discount,177,0.02
7,Financial Adjustment,151,0.01
8,Sample,104,0.01
9,Test Transaction,17,0.00


### Make sure we haven't lost rows

In [14]:
classified_rows = (
    transaction_type_summary["rows"].sum()
)

print(
    f"Original rows:   {len(raw_df):,}"
)

print(
    f"Classified rows: {classified_rows:,}"
)

print(
    "Reconciliation:",
    len(raw_df) == classified_rows
)

Original rows:   1,067,371
Classified rows: 1,067,371
Reconciliation: True


### Inspect Other / Review

In [15]:
other_review = df[
    df["transaction_type"] ==
    "Other / Review"
]

print(
    "Rows requiring additional review:",
    f"{len(other_review):,}"
)

display(
    other_review[
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "unit_price",
            "customer_id",
            "country",
            "transaction_type"
        ]
    ].head(50)
)

Rows requiring additional review: 0


,invoice,stock_code,description,quantity,unit_price,customer_id,country,transaction_type


### Validate known findings survived transformation

In [16]:
validation = {
    "Total Rows":
        len(df),

    "Cancellation Rows":
        df["is_cancellation"].sum(),

    "Negative Quantity Rows":
        df["is_negative_quantity"].sum(),

    "Zero Price Rows":
        df["is_zero_price"].sum(),

    "Negative Price Rows":
        df["is_negative_price"].sum(),

    "Missing Customer Rows":
        df["is_missing_customer"].sum(),

    "Missing Description Rows":
        df["is_missing_description"].sum(),

    "Exact Duplicate Rows":
        df["is_exact_duplicate"].sum()
}

validation_df = pd.DataFrame(
    validation.items(),
    columns=[
        "Check",
        "Count"
    ]
)

display(validation_df)

,Check,Count
0,Total Rows,1067371
1,Cancellation Rows,19494
2,Negative Quantity Rows,22950
3,Zero Price Rows,6202
4,Negative Price Rows,5
5,Missing Customer Rows,243007
6,Missing Description Rows,4382
7,Exact Duplicate Rows,34335


### Resolve the duplicate question

In [17]:
# 1. Exact duplicates in the untouched combined source
raw_exact_duplicates = raw_df.duplicated(
    keep="first"
).sum()

# 2. Duplicates after normalization, but treating
# each source sheet independently
normalized_with_sheet_duplicates = df.duplicated(
    subset=duplicate_columns + ["source_sheet"],
    keep="first"
).sum()

# 3. Duplicates after normalization across the
# entire two-year dataset
normalized_cross_sheet_duplicates = df.duplicated(
    subset=duplicate_columns,
    keep="first"
).sum()

print(
    "Raw exact duplicates:",
    f"{raw_exact_duplicates:,}"
)

print(
    "Normalized duplicates WITH source sheet:",
    f"{normalized_with_sheet_duplicates:,}"
)

print(
    "Normalized duplicates WITHOUT source sheet:",
    f"{normalized_cross_sheet_duplicates:,}"
)

Raw exact duplicates: 12,133
Normalized duplicates WITH source sheet: 12,133
Normalized duplicates WITHOUT source sheet: 34,335


### Check each worksheet's date range

In [18]:
sheet_date_ranges = (
    df.groupby("source_sheet")
      .agg(
          rows=("invoice", "size"),
          min_date=("invoice_date", "min"),
          max_date=("invoice_date", "max")
      )
)

display(sheet_date_ranges)

,rows,min_date,max_date
source_sheet,,,
Year 2009-2010,525461,2009-12-01 07:45:00,2010-12-09 20:01:00
Year 2010-2011,541910,2010-12-01 08:26:00,2011-12-09 12:50:00


In [19]:
overlap_start = (
    sheet_date_ranges["min_date"].max()
)

overlap_end = (
    sheet_date_ranges["max_date"].min()
)

print("Possible overlap start:", overlap_start)
print("Possible overlap end:  ", overlap_end)

if overlap_start <= overlap_end:
    print("The worksheets overlap in time.")
else:
    print("The worksheets do not overlap in time.")

Possible overlap start: 2010-12-01 08:26:00
Possible overlap end:   2010-12-09 20:01:00
The worksheets overlap in time.


### Identify records appearing across both worksheets

In [20]:
temp = df[
    duplicate_columns + ["source_sheet"]
].copy()

temp["transaction_hash"] = (
    pd.util.hash_pandas_object(
        temp[duplicate_columns],
        index=False
    )
)

hash_sheet_counts = (
    temp.groupby("transaction_hash")
        ["source_sheet"]
        .nunique()
)

cross_sheet_hashes = hash_sheet_counts[
    hash_sheet_counts > 1
].index

cross_sheet_rows = temp[
    temp["transaction_hash"].isin(
        cross_sheet_hashes
    )
]

print(
    "Unique transaction patterns appearing "
    "in multiple sheets:",
    f"{len(cross_sheet_hashes):,}"
)

print(
    "Total rows belonging to those patterns:",
    f"{len(cross_sheet_rows):,}"
)

Unique transaction patterns appearing in multiple sheets: 22,202
Total rows belonging to those patterns: 45,046


In [21]:
display(
    cross_sheet_rows
    .sort_values(
        [
            "invoice_date",
            "invoice",
            "stock_code"
        ]
    )
    .head(30)
)

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,source_sheet,transaction_hash
502944,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,Year 2009-2010,14980842894886436539
525467,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,Year 2010-2011,14980842894886436539
502943,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,Year 2009-2010,10773825968856796282
525466,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,Year 2010-2011,10773825968856796282
502939,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2009-2010,5138431326373849053
525462,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2010-2011,5138431326373849053
502942,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2009-2010,16773037220223553723
525465,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2010-2011,16773037220223553723
502941,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2009-2010,13007158363834338916
525464,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2010-2011,13007158363834338916


# Build the analytical layers

### Create detailed duplicate flags

In [22]:
# Exact business fields that define one transaction line
business_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country"
]

# Hash used only for audit / duplicate investigation
df["transaction_hash"] = pd.util.hash_pandas_object(
    df[business_columns],
    index=False
).astype("uint64")

# Duplicate inside the same source worksheet
df["is_within_sheet_duplicate"] = df.duplicated(
    subset=business_columns + ["source_sheet"],
    keep="first"
)

# Duplicate across the complete workbook
df["is_analytical_duplicate"] = df.duplicated(
    subset=business_columns,
    keep="first"
)

# Transaction patterns occurring in more than one worksheet
sheet_count_per_hash = (
    df.groupby("transaction_hash")["source_sheet"]
      .transform("nunique")
)

df["appears_across_sheets"] = (
    sheet_count_per_hash > 1
)

print(
    "Within-sheet duplicates:",
    f"{df['is_within_sheet_duplicate'].sum():,}"
)

print(
    "Analytical duplicates:",
    f"{df['is_analytical_duplicate'].sum():,}"
)

print(
    "Rows belonging to cross-sheet patterns:",
    f"{df['appears_across_sheets'].sum():,}"
)

Within-sheet duplicates: 12,133
Analytical duplicates: 34,335
Rows belonging to cross-sheet patterns: 45,046


### Create the deduplicated analytical master

In [23]:
analytics_df = (
    df[
        ~df["is_analytical_duplicate"]
    ]
    .copy()
)

print(
    f"Normalized source rows: "
    f"{len(df):,}"
)

print(
    f"Removed duplicate occurrences: "
    f"{df['is_analytical_duplicate'].sum():,}"
)

print(
    f"Analytical rows: "
    f"{len(analytics_df):,}"
)

print(
    "Reconciliation:",
    len(df)
    ==
    len(analytics_df)
    + df["is_analytical_duplicate"].sum()
)

Normalized source rows: 1,067,371
Removed duplicate occurrences: 34,335
Analytical rows: 1,033,036
Reconciliation: True


### See how much duplicates would have affected sales

In [24]:
sales_before_dedup = df[
    df["transaction_type"] ==
    "Merchandise Sale"
].copy()

sales_after_dedup = analytics_df[
    analytics_df["transaction_type"] ==
    "Merchandise Sale"
].copy()

revenue_before = (
    sales_before_dedup["quantity"]
    * sales_before_dedup["unit_price"]
).sum()

revenue_after = (
    sales_after_dedup["quantity"]
    * sales_after_dedup["unit_price"]
).sum()

duplicate_revenue_impact = (
    revenue_before - revenue_after
)

print(
    f"Merchandise revenue before dedup: "
    f"£{revenue_before:,.2f}"
)

print(
    f"Merchandise revenue after dedup:  "
    f"£{revenue_after:,.2f}"
)

print(
    f"Potential duplicate overstatement: "
    f"£{duplicate_revenue_impact:,.2f}"
)

print(
    f"Revenue impact percentage: "
    f"{duplicate_revenue_impact / revenue_before * 100:.2f}%"
)

Merchandise revenue before dedup: £20,111,952.24
Merchandise revenue after dedup:  £19,645,632.86
Potential duplicate overstatement: £466,319.38
Revenue impact percentage: 2.32%


### Create Merchandise Sales layer

In [25]:
merchandise_sales = analytics_df[
    analytics_df["transaction_type"]
    == "Merchandise Sale"
].copy()

merchandise_sales["gross_revenue"] = (
    merchandise_sales["quantity"]
    * merchandise_sales["unit_price"]
)

print(
    "Merchandise sales rows:",
    f"{len(merchandise_sales):,}"
)

print(
    "Gross merchandise revenue:",
    f"£{merchandise_sales['gross_revenue'].sum():,.2f}"
)

Merchandise sales rows: 1,003,439
Gross merchandise revenue: £19,645,632.86


### Create Customer Sales layer

In [26]:
customer_sales = merchandise_sales[
    merchandise_sales["customer_id"].notna()
].copy()

print(
    "Merchandise sales rows:",
    f"{len(merchandise_sales):,}"
)

print(
    "Customer-identifiable sales rows:",
    f"{len(customer_sales):,}"
)

print(
    "Sales rows without customer ID:",
    f"{merchandise_sales['customer_id'].isna().sum():,}"
)

print(
    "Unique identifiable customers:",
    f"{customer_sales['customer_id'].nunique():,}"
)

Merchandise sales rows: 1,003,439
Customer-identifiable sales rows: 776,596
Sales rows without customer ID: 226,843
Unique identifiable customers: 5,852


### Create cancellation / returns layer

In [27]:
returns_cancellations = analytics_df[
    analytics_df["transaction_type"]
    == "Merchandise Cancellation"
].copy()

returns_cancellations["cancelled_units"] = (
    returns_cancellations["quantity"].abs()
)

returns_cancellations["cancelled_value"] = (
    returns_cancellations["cancelled_units"]
    * returns_cancellations["unit_price"]
)

print(
    "Cancellation rows:",
    f"{len(returns_cancellations):,}"
)

print(
    "Unique cancellation invoices:",
    f"{returns_cancellations['invoice'].nunique():,}"
)

print(
    "Cancelled units:",
    f"{returns_cancellations['cancelled_units'].sum():,.0f}"
)

print(
    "Cancellation value:",
    f"£{returns_cancellations['cancelled_value'].sum():,.2f}"
)

Cancellation rows: 17,932
Unique cancellation invoices: 7,423
Cancelled units: 467,757
Cancellation value: £724,465.56


### Create inventory/operational adjustment layer

In [28]:
inventory_adjustments = analytics_df[
    analytics_df["transaction_type"].isin(
        [
            "Inventory / Operational Adjustment",
            "Manual / System Adjustment"
        ]
    )
].copy()

print(
    "Inventory / operational adjustment rows:",
    f"{len(inventory_adjustments):,}"
)

display(
    inventory_adjustments[
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "unit_price",
            "invoice_date"
        ]
    ].head(20)
)

Inventory / operational adjustment rows: 4,850


,invoice,stock_code,description,quantity,unit_price,invoice_date
263,489464,21733,85123a mixed,-96,0.00,2009-12-01 10:52:00
283,489463,71477,short,-240,0.00,2009-12-01 10:52:00
284,489467,85123A,21733 mixed,-192,0.00,2009-12-01 10:53:00
470,489521,21646,<NA>,-50,0.00,2009-12-01 11:44:00
2697,489609,M,Manual,1,4.00,2009-12-01 14:50:00
3053,C489651,M,Manual,-1,5.10,2009-12-01 16:48:00
3114,489655,20683,<NA>,-44,0.00,2009-12-01 17:26:00
3162,489660,35956,lost,-1043,0.00,2009-12-01 17:43:00
3168,489663,35605A,damages,-117,0.00,2009-12-01 18:02:00
4296,489806,18010,<NA>,-770,0.00,2009-12-02 12:42:00


### Create financial adjustment layer

In [29]:
financial_adjustments = analytics_df[
    analytics_df["transaction_type"].isin(
        [
            "Financial Adjustment",
            "Discount"
        ]
    )
].copy()

print(
    "Financial adjustment rows:",
    f"{len(financial_adjustments):,}"
)

display(
    financial_adjustments[
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "unit_price",
            "transaction_value"
        ]
    ].head(20)
)

Financial adjustment rows: 315


,invoice,stock_code,description,quantity,unit_price,transaction_value
735,C489535,D,Discount,-1,9.00,-9.00
736,C489535,D,Discount,-1,19.00,-19.00
18410,C490943,BANK CHARGES,Bank Charges,-1,15.00,-15.00
18466,490948,BANK CHARGES,Bank Charges,1,15.00,15.00
24675,C491428,D,Discount,-1,9.10,-9.10
29414,C491845,D,Discount,-1,1.59,-1.59
29958,C491962,D,Discount,-1,0.59,-0.59
33435,C492206,BANK CHARGES,Bank Charges,-1,848.43,-848.43
39127,C492693,D,Discount,-1,6.85,-6.85
44782,C493373,D,Discount,-1,64.37,-64.37


### Create non-revenue product movement layer

In [30]:
zero_price_movements = analytics_df[
    analytics_df["transaction_type"]
    == "Zero-Price Product Movement"
].copy()

print(
    "Zero-price product movement rows:",
    f"{len(zero_price_movements):,}"
)

print(
    "With Customer ID:",
    f"{zero_price_movements['customer_id'].notna().sum():,}"
)

print(
    "Units moved:",
    f"{zero_price_movements['quantity'].sum():,.0f}"
)

Zero-price product movement rows: 2,594
With Customer ID: 61
Units moved: 243,836


### Create Inventory Demand layer

In [31]:
inventory_demand = merchandise_sales.copy()

inventory_demand["demand_units"] = (
    inventory_demand["quantity"]
)

print(
    "Inventory demand rows:",
    f"{len(inventory_demand):,}"
)

print(
    "Total merchandise demand units:",
    f"{inventory_demand['demand_units'].sum():,.0f}"
)

print(
    "Unique merchandise products:",
    f"{inventory_demand['stock_code'].nunique():,}"
)

Inventory demand rows: 1,003,439
Total merchandise demand units: 11,188,146
Unique merchandise products: 4,903


### Analytical-layer reconciliation

In [32]:
layer_summary = (
    analytics_df["transaction_type"]
    .value_counts()
    .rename_axis("transaction_type")
    .reset_index(name="rows")
)

layer_summary["percentage"] = (
    layer_summary["rows"]
    / len(analytics_df)
    * 100
).round(2)

display(layer_summary)

print(
    "\nAnalytical master rows:",
    f"{len(analytics_df):,}"
)

print(
    "Classification total:",
    f"{layer_summary['rows'].sum():,}"
)

print(
    "Reconciliation:",
    len(analytics_df)
    ==
    layer_summary["rows"].sum()
)

,transaction_type,rows,percentage
0,Merchandise Sale,1003439,97.13
1,Merchandise Cancellation,17932,1.74
2,Shipping / Service Charge,3788,0.37
3,Inventory / Operational Adjustment,3393,0.33
4,Zero-Price Product Movement,2594,0.25
5,Manual / System Adjustment,1457,0.14
6,Discount,173,0.02
7,Financial Adjustment,142,0.01
8,Sample,101,0.01
9,Test Transaction,17,0.00



Analytical master rows: 1,033,036
Classification total: 1,033,036
Reconciliation: True


### Final source → analytical reconciliation

In [33]:
etl_reconciliation = pd.DataFrame({
    "Metric": [
        "Raw Source Rows",
        "Exact Duplicate Occurrences Removed",
        "Final Analytical Rows",
        "Merchandise Sales",
        "Customer-Identifiable Sales",
        "Merchandise Cancellations",
        "Operational / Inventory Adjustments",
        "Financial Adjustments",
        "Zero-Price Product Movements"
    ],

    "Count": [
        len(df),
        df["is_analytical_duplicate"].sum(),
        len(analytics_df),
        len(merchandise_sales),
        len(customer_sales),
        len(returns_cancellations),
        len(inventory_adjustments),
        len(financial_adjustments),
        len(zero_price_movements)
    ]
})

display(etl_reconciliation)

,Metric,Count
0,Raw Source Rows,1067371
1,Exact Duplicate Occurrences Removed,34335
2,Final Analytical Rows,1033036
3,Merchandise Sales,1003439
4,Customer-Identifiable Sales,776596
5,Merchandise Cancellations,17932
6,Operational / Inventory Adjustments,4850
7,Financial Adjustments,315
8,Zero-Price Product Movements,2594


### Locate/create interim folder

In [34]:
from pathlib import Path

current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nInterim data directory:")
print(INTERIM_DIR)

Project root:
c:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization

Interim data directory:
c:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\data\interim


### Select columns for the analytical master

In [35]:
master_columns = [
    # Source / identifiers
    "invoice",
    "stock_code",
    "description",
    "customer_id",
    "country",
    "source_sheet",

    # Transaction values
    "quantity",
    "unit_price",
    "transaction_value",
    "invoice_date",

    # Calendar
    "invoice_year",
    "invoice_month",
    "year_month",
    "invoice_week",
    "day_of_week",

    # Business classification
    "transaction_type",

    # Data-quality flags
    "is_cancellation",
    "is_negative_quantity",
    "is_zero_price",
    "is_negative_price",
    "is_missing_customer",
    "is_missing_description",
    "is_exact_duplicate",
    "is_within_sheet_duplicate",
    "is_analytical_duplicate",
    "appears_across_sheets",

    # Audit identifier
    "transaction_hash"
]

analytics_master = analytics_df[
    master_columns
].copy()

print(
    "Analytics master shape:",
    analytics_master.shape
)

display(
    analytics_master.head()
)

Analytics master shape: (1033036, 27)


,invoice,stock_code,description,customer_id,country,source_sheet,quantity,unit_price,transaction_value,invoice_date,invoice_year,invoice_month,year_month,invoice_week,day_of_week,transaction_type,is_cancellation,is_negative_quantity,is_zero_price,is_negative_price,is_missing_customer,is_missing_description,is_exact_duplicate,is_within_sheet_duplicate,is_analytical_duplicate,appears_across_sheets,transaction_hash
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.95,83.40,2009-12-01 07:45:00,2009,12,2009-12,2009-11-30/2009-12-06,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,False,False,False,7243821383875921504
1,489434,79323P,PINK CHERRY LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.75,81.00,2009-12-01 07:45:00,2009,12,2009-12,2009-11-30/2009-12-06,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,False,False,False,17514895690039914094
2,489434,79323W,WHITE CHERRY LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.75,81.00,2009-12-01 07:45:00,2009,12,2009-12,2009-11-30/2009-12-06,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,False,False,False,12457407259001370697
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",13085,United Kingdom,Year 2009-2010,48,2.10,100.80,2009-12-01 07:45:00,2009,12,2009-12,2009-11-30/2009-12-06,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,False,False,False,18012193762395552222
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,13085,United Kingdom,Year 2009-2010,24,1.25,30.00,2009-12-01 07:45:00,2009,12,2009-12,2009-11-30/2009-12-06,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,False,False,False,8093434425875725190


### Save analytical master

In [36]:
import sys
import pandas as pd
import pyarrow as pa

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("PyArrow location:", pa.__file__)

Python: c:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\.venv\Scripts\python.exe
Pandas: 3.0.5
PyArrow: 25.0.1
PyArrow location: c:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\.venv\Lib\site-packages\pyarrow\__init__.py


In [37]:
analytics_master.to_parquet(
    INTERIM_DIR / "analytics_master.parquet",
    index=False
)

print("Saved analytics_master.parquet")

Saved analytics_master.parquet


### Save merchandise sales

In [38]:
merchandise_sales.to_parquet(
    INTERIM_DIR / "merchandise_sales.parquet",
    index=False
)

print(
    "Saved merchandise_sales.parquet:",
    f"{len(merchandise_sales):,} rows"
)

Saved merchandise_sales.parquet: 1,003,439 rows


### Save customer sales

In [39]:
customer_sales.to_parquet(
    INTERIM_DIR / "customer_sales.parquet",
    index=False
)

print(
    "Saved customer_sales.parquet:",
    f"{len(customer_sales):,} rows"
)

Saved customer_sales.parquet: 776,596 rows


### Save cancellations

In [40]:
returns_cancellations.to_parquet(
    INTERIM_DIR / "returns_cancellations.parquet",
    index=False
)

print(
    "Saved returns_cancellations.parquet:",
    f"{len(returns_cancellations):,} rows"
)

Saved returns_cancellations.parquet: 17,932 rows


### Save operational adjustments

In [41]:
inventory_adjustments.to_parquet(
    INTERIM_DIR / "operational_adjustments.parquet",
    index=False
)

print(
    "Saved operational_adjustments.parquet:",
    f"{len(inventory_adjustments):,} rows"
)

Saved operational_adjustments.parquet: 4,850 rows


### Save financial adjustments

In [42]:
financial_adjustments.to_parquet(
    INTERIM_DIR / "financial_adjustments.parquet",
    index=False
)

print(
    "Saved financial_adjustments.parquet:",
    f"{len(financial_adjustments):,} rows"
)

Saved financial_adjustments.parquet: 315 rows


### Save zero-price product movements

In [43]:
zero_price_movements.to_parquet(
    INTERIM_DIR / "zero_price_movements.parquet",
    index=False
)

print(
    "Saved zero_price_movements.parquet:",
    f"{len(zero_price_movements):,} rows"
)

Saved zero_price_movements.parquet: 2,594 rows


### Save inventory-demand layer

In [44]:
inventory_demand.to_parquet(
    INTERIM_DIR / "inventory_demand.parquet",
    index=False
)

print(
    "Saved inventory_demand.parquet:",
    f"{len(inventory_demand):,} rows"
)

Saved inventory_demand.parquet: 1,003,439 rows


### Save ETL reconciliation

In [45]:
etl_reconciliation.to_csv(
    INTERIM_DIR / "etl_reconciliation.csv",
    index=False
)

print("Saved etl_reconciliation.csv")

Saved etl_reconciliation.csv


### Verify every output file

In [46]:
output_files = sorted(
    INTERIM_DIR.iterdir()
)

print("Files created:\n")

for file in output_files:
    
    size_mb = (
        file.stat().st_size /
        (1024 ** 2)
    )
    
    print(
        f"{file.name:<40} "
        f"{size_mb:>8.2f} MB"
    )

Files created:

analytics_master.parquet                    16.67 MB
customer_sales.parquet                      13.26 MB
etl_reconciliation.csv                       0.00 MB
financial_adjustments.parquet                0.03 MB
inventory_demand.parquet                    17.94 MB
merchandise_sales.parquet                   17.11 MB
operational_adjustments.parquet              0.18 MB
returns_cancellations.parquet                0.56 MB
zero_price_movements.parquet                 0.09 MB


### Reload test

In [47]:
test_load = pd.read_parquet(
    INTERIM_DIR /
    "analytics_master.parquet"
)

print(
    "Reloaded rows:",
    f"{len(test_load):,}"
)

print(
    "Reloaded columns:",
    test_load.shape[1]
)

print(
    "Row-count match:",
    len(test_load) == len(analytics_master)
)

Reloaded rows: 1,033,036
Reloaded columns: 27
Row-count match: True


In [48]:
del test_load

### Create a small data-quality summary

In [49]:
data_quality_summary = pd.DataFrame({
    "Metric": [
        "Raw Source Rows",
        "Missing Customer ID Rows",
        "Missing Description Rows",
        "Cancellation Rows",
        "Negative Quantity Rows",
        "Zero Price Rows",
        "Negative Price Rows",
        "Analytical Duplicate Occurrences",
        "Final Analytical Rows",
        "Duplicate Revenue Overstatement",
        "Duplicate Revenue Impact %"
    ],

    "Value": [
        f"{len(df):,}",
        f"{df['is_missing_customer'].sum():,}",
        f"{df['is_missing_description'].sum():,}",
        f"{df['is_cancellation'].sum():,}",
        f"{df['is_negative_quantity'].sum():,}",
        f"{df['is_zero_price'].sum():,}",
        f"{df['is_negative_price'].sum():,}",
        f"{df['is_analytical_duplicate'].sum():,}",
        f"{len(analytics_df):,}",
        f"£{duplicate_revenue_impact:,.2f}",
        f"{duplicate_revenue_impact / revenue_before * 100:.2f}%"
    ]
})

display(data_quality_summary)

data_quality_summary.to_csv(
    INTERIM_DIR /
    "data_quality_summary.csv",
    index=False
)

,Metric,Value
0,Raw Source Rows,"1,067,371"
1,Missing Customer ID Rows,"243,007"
2,Missing Description Rows,"4,382"
3,Cancellation Rows,"19,494"
4,Negative Quantity Rows,"22,950"
5,Zero Price Rows,"6,202"
6,Negative Price Rows,5
7,Analytical Duplicate Occurrences,"34,335"
8,Final Analytical Rows,"1,033,036"
9,Duplicate Revenue Overstatement,"£466,319.38"
